In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset

import shap 

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tqdm import tqdm

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *

from autogluon.tabular import TabularPredictor

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/home/sr100@hsg.privat/x-med/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [2]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)
#forward_fill_imputation(ts_data) 

#notes = create_notes_df(dfs, filename='../data/embeddings/emb_gte.npy')
notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')
#notes = create_notes_df(dfs, filename=None)

full_dataset = NephroCAGEDataset(static_df=static_df, ts_data=ts_data, notes_df=notes, biopsy_df=dfs['biopsy'])
datapoints_limit = len(full_dataset)
#datapoints_limit = 320
dataset = Subset(full_dataset, indices=list(range(datapoints_limit)))
ts_scaler = full_dataset.ts_scaler
static_scaler = full_dataset.scaler

Unique patients in medication: 3335
Removing patients that are not in static_df
Unique patients in clinical assessments: 3296
Average entries per patient 78.9
Unique patients in clinical assessments: 3465
Removing patients that are not in static_df
Unique patients in clinical assessments: 3423
Average entries per patient 61.16155419222904
Unique patients in lab df: 3460
Removing patients that are not in static_df
Unique patients in lab df: 3410
Average entries per patient 472.2140762463343
Reading notes from exams.csv
Found 206509 texts
Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9
3382 patients remain after removing no-notes patients.



In [ ]:
# Split dataset into training and test sets (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

#vanilla_lstm = VanillaTimeSeriesEncoder()
att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(att_encoder, categorical_cardinalities=full_dataset.categorical_cardinalities, use_static=True, use_notes=True).to(device)
predict_steps_ahead = 1

model.load_state_dict(torch.load('../models/after10epochs.pt', weights_only=True))

/home/sr100@hsg.privat/x-med/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


<All keys matched successfully>

In [4]:
def extract_horizon_reprs(
    dataloader, 
    model, 
    horizons, 
    label_key, 
    rel_days_key=None,
    min_history_days=90,
    max_days=180,
    max_samples_per_patient=5
):
    """
    - Feeds each patient's entire time series (minus last step) in one pass.
    - Extracts hidden representations for each time step from model output.
    - For each horizon H, determines if the event occurs within H days from that step.
    - Only includes time steps between min_history_days and max_days.
    - Takes up to max_samples_per_patient samples per patient (first N valid samples).

    If `label_key` corresponds to a multi-day event list (like "rej_rel_days"), then `rel_days_key`
    can be left None (ignored), and we will handle the logic differently.
    
    Parameters:
    -----------
    dataloader : DataLoader
        The PyTorch dataloader yielding patient data batches
    model : torch.nn.Module
        The trained model to extract representations from
    horizons : list
        List of horizon values (in days) to consider
    label_key : str
        Key in batch dictionary for labels
    rel_days_key : str, optional
        Key for relative days to event (only for single-event labels)
    min_history_days : int, default=90
        Minimum number of days of history required
    max_days : int, default=180
        Maximum number of days to include in the dataset
    max_samples_per_patient : int, default=5
        Maximum number of samples to include per patient
    """
    from tqdm import tqdm
    
    model.eval()

    # For each horizon, prepare storage for hidden reps, labels, day-of-step, patient_id
    hr_repr = {H: [] for H in horizons}
    hr_label = {H: [] for H in horizons}
    hr_days  = {H: [] for H in horizons}
    hr_pids  = {H: [] for H in horizons}
    
    # Keep track of samples per patient for each horizon
    patient_sample_counts = {H: {} for H in horizons}

    all_pids = set()
    positive_pids = set()

    with torch.no_grad():
        # Add progress bar for the dataloader iteration
        for batch in tqdm(dataloader, desc="Processing patients"):
            pid  = batch['patient_id']
            slen = batch['seq_len']

            # Static features
            cat_static = batch['static_categorical_features'].to(device)
            num_static = batch['static_numerical_features'].to(device)

            # Time series
            full_ts    = batch['ts_features'].to(device)  # shape (B, T, F)
            timesteps  = batch['timesteps'].to(device)    # shape (B, T)
            mask_      = batch['mask'].to(device)         # shape (B, T)

            # --- Convert single-element Tensors in label_key to float/list ---
            raw_labels_data = batch[label_key]  # shape (B,)
            labels_data = []
            for val in raw_labels_data:
                if isinstance(val, torch.Tensor):
                    # If it's a single element, convert to float
                    if val.numel() == 1:
                        val = float(val.item())
                    else:
                        # If multi-element, convert to NumPy or list
                        val = val.cpu().numpy()
                # Otherwise, val can be float, int, list, etc.
                labels_data.append(val)

            # If single-event usage, we also have rel_days_key => shape (B,)
            if rel_days_key and rel_days_key in batch:
                raw_rel_days_data = batch[rel_days_key]
                rel_days_data = []
                for dval in raw_rel_days_data:
                    if isinstance(dval, torch.Tensor):
                        if dval.numel() == 1:
                            dval = float(dval.item())
                        else:
                            dval = dval.cpu().numpy()
                    rel_days_data.append(dval)
                rel_days_data = np.array(rel_days_data)
            else:
                rel_days_data = None  # We'll handle multi-day logic below

            # If using notes
            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps  = batch['notes_timesteps'].to(device)
            notes_mask       = batch['notes_mask'].to(device)

            B, T, F = full_ts.shape

            # Loop over each patient in this batch
            for i in range(B):
                patient_id_i = pid[i]
                all_pids.add(patient_id_i)

                # label_or_list => single-event (float 0/1) or multi-event list
                label_or_list = labels_data[i]

                # Keep track of which patients have at least one event
                if isinstance(label_or_list, (int, float, np.number)):
                    # single label
                    if label_or_list == 1:
                        positive_pids.add(patient_id_i)
                elif isinstance(label_or_list, (list, np.ndarray)):
                    # multiple days => if not empty => event
                    if len(label_or_list) > 0:
                        positive_pids.add(patient_id_i)
                elif label_or_list is not None:
                    # Unknown type
                    raise TypeError(f"Unsupported label type: {type(label_or_list)}")

                # If the sequence is too short
                if slen[i] < 2:
                    continue
                
                # Check if we already have max samples for this patient for all horizons
                if max_samples_per_patient > 0:
                    all_horizons_at_max = True
                    for H in horizons:
                        if patient_id_i not in patient_sample_counts[H] or patient_sample_counts[H][patient_id_i] < max_samples_per_patient:
                            all_horizons_at_max = False
                            break
                    
                    if all_horizons_at_max:
                        continue  # Skip this patient altogether if already at max for all horizons

                # Slice the valid portion of the time series: [0..slen[i]-1]
                seq_len_i = slen[i].item()
                ts_i = full_ts[i:i+1, :seq_len_i, :]   # (1, seq_len_i, F)
                tm_i = timesteps[i:i+1, :seq_len_i]    # (1, seq_len_i)
                mk_i = mask_[i:i+1, :seq_len_i]        # (1, seq_len_i)

                # Model input => omit last step from time series
                inp_seq = ts_i[:, :-1, :]              # (1, seq_len_i-1, F)
                elapsed_times = tm_i[:, 1:] - tm_i[:, :-1]
                inp_mask = mk_i[:, :-1]

                notes_emb_i = notes_embeddings[i:i+1]
                notes_ts_i  = notes_timesteps[i:i+1]
                notes_mk_i  = notes_mask[i:i+1]

                # Forward pass
                out, lstm_out, _, _ = model(
                    x=inp_seq,
                    elapsed_times=elapsed_times,     # (1, seq_len_i - 1)
                    timesteps=tm_i[:, :-1],          # (1, seq_len_i - 1)
                    notes_embeddings=notes_emb_i,
                    notes_timesteps=notes_ts_i,
                    static_features=(cat_static[i:i+1], num_static[i:i+1]),
                    mask=inp_mask,                   # (1, seq_len_i - 1)
                    notes_mask=notes_mk_i
                )
                # lstm_out => shape (1, seq_len_i-1, hidden_size)

                # Time array for all steps
                time_arr = tm_i.cpu().numpy().flatten()  # (seq_len_i,)
                
                # Initialize counts for this patient if not present
                for H in horizons:
                    if patient_id_i not in patient_sample_counts[H]:
                        patient_sample_counts[H][patient_id_i] = 0
                
                # Loop through time steps and only process until we reach max_samples for each horizon
                for k in range(seq_len_i - 1):
                    cur_day = time_arr[k+1]  # day of the (k+1)-th step
                    
                    # Skip if outside the desired range
                    if cur_day < min_history_days or cur_day > max_days:
                        continue
                    
                    # Get representation for this time step
                    rep_ = lstm_out[0, k, :].cpu().numpy()
                    
                    # Check for each horizon if we still need more samples
                    any_horizon_needs_samples = False
                    for H in horizons:
                        if patient_sample_counts[H][patient_id_i] < max_samples_per_patient:
                            any_horizon_needs_samples = True
                            break
                    
                    if not any_horizon_needs_samples:
                        break  # Exit time step loop if all horizons have enough samples
                    
                    # Process for each horizon that still needs samples
                    for H in horizons:
                        # Skip if already have max samples for this horizon
                        if patient_sample_counts[H][patient_id_i] >= max_samples_per_patient:
                            continue
                        
                        # If single-event logic is in play
                        if rel_days_data is not None:
                            event_label = label_or_list     # 0 or 1
                            event_day   = rel_days_data[i]  # single day
                            if (event_label == 1) and (0 < (event_day - cur_day) <= H):
                                label_ = 1
                            else:
                                label_ = 0

                        # Else multi-event logic (like rejections)
                        else:
                            # label_or_list is a list of event days or None
                            if isinstance(label_or_list, (list, np.ndarray)) and len(label_or_list) > 0:
                                # label_ = 1 if any day d in label_or_list satisfies (0 < d - cur_day <= H)
                                label_ = int(any(0 < (d - cur_day) <= H for d in label_or_list))
                            else:
                                label_ = 0

                        # Store
                        hr_repr[H].append(rep_)
                        hr_label[H].append(label_)
                        hr_days[H].append(cur_day)
                        hr_pids[H].append(patient_id_i)
                        
                        # Update sample count
                        patient_sample_counts[H][patient_id_i] += 1

    # Convert lists to numpy arrays for convenience
    for H in horizons:
        hr_repr[H] = np.array(hr_repr[H])
        hr_label[H] = np.array(hr_label[H])
        hr_days[H]  = np.array(hr_days[H])
        hr_pids[H]  = np.array(hr_pids[H])

    # Calculate and print statistics
    for H in horizons:
        total_patients = len(patient_sample_counts[H])
        avg_samples = np.mean([patient_sample_counts[H][pid] for pid in patient_sample_counts[H]])
        max_samples = max([patient_sample_counts[H][pid] for pid in patient_sample_counts[H]]) if patient_sample_counts[H] else 0
        
        print(f"Horizon {H}: {total_patients} patients, avg {avg_samples:.1f} samples/patient, max {max_samples} samples/patient")

    print(f"Total unique patients: {len(all_pids)}, patients with event: {len(positive_pids)}")
    return hr_repr, hr_label, hr_days, hr_pids

In [5]:
horizons = [30, 90, 180]
min_history_days = 90
max_days = 720

print("Embedding patients...")

train_graft_repr, train_graft_lbl, train_graft_days, train_graft_pids = extract_horizon_reprs(train_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days)
test_graft_repr, test_graft_lbl, test_graft_days, test_graft_pids = extract_horizon_reprs(test_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days)

train_rej_repr, train_rej_lbl, train_rej_days, train_rej_pids = extract_horizon_reprs(train_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days)
test_rej_repr, test_rej_lbl, test_rej_days, test_rej_pids = extract_horizon_reprs(test_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days)

train_mort_repr, train_mort_lbl, train_mort_days, train_mort_pids = extract_horizon_reprs(train_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days)
test_mort_repr, test_mort_lbl, test_mort_days, test_mort_pids = extract_horizon_reprs(test_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days)

# Print some stats about the extracted data
for H in horizons:
    print(f"\n===== Extracted Features for Horizon {H} days =====")
    print(f"Graft Loss Training: {train_graft_repr[H].shape}, Positive: {sum(train_graft_lbl[H])}, Ratio: {sum(train_graft_lbl[H])/len(train_graft_lbl[H]):.4f}")
    print(f"Graft Loss Testing: {test_graft_repr[H].shape}, Positive: {sum(test_graft_lbl[H])}, Ratio: {sum(test_graft_lbl[H])/len(test_graft_lbl[H]):.4f}")
    print(f"Rejection Training: {train_rej_repr[H].shape}, Positive: {sum(train_rej_lbl[H])}, Ratio: {sum(train_rej_lbl[H])/len(train_rej_lbl[H]):.4f}")
    print(f"Rejection Testing: {test_rej_repr[H].shape}, Positive: {sum(test_rej_lbl[H])}, Ratio: {sum(test_rej_lbl[H])/len(test_rej_lbl[H]):.4f}")

Embedding patients...


Processing patients: 100%|██████████| 170/170 [04:29<00:00,  1.58s/it]


Horizon 30: 2705 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 2705 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 2705 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 2705, patients with event: 467


Processing patients: 100%|██████████| 43/43 [01:05<00:00,  1.53s/it]


Horizon 30: 677 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 677 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 677 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 677, patients with event: 119


...

Horizon 30: 677 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 677 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 677 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 677, patients with event: 54


Processing patients: 100%|██████████| 170/170 [04:30<00:00,  1.59s/it]


Horizon 30: 2705 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 2705 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 2705 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 2705, patients with event: 821


Processing patients: 100%|██████████| 43/43 [01:06<00:00,  1.55s/it]

Horizon 30: 677 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 677 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 677 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 677, patients with event: 217

===== Extracted Features for Horizon 30 days =====
Graft Loss Training: (12926, 512), Positive: 29, Ratio: 0.0022
Graft Loss Testing: (3272, 512), Positive: 6, Ratio: 0.0018
Rejection Training: (12926, 512), Positive: 127, Ratio: 0.0098
Rejection Testing: (3272, 512), Positive: 31, Ratio: 0.0095

===== Extracted Features for Horizon 90 days =====
Graft Loss Training: (12926, 512), Positive: 89, Ratio: 0.0069
Graft Loss Testing: (3272, 512), Positive: 24, Ratio: 0.0073
Rejection Training: (12926, 512), Positive: 208, Ratio: 0.0161
Rejection Testing: (3272, 512), Positive: 50, Ratio: 0.0153

===== Extracted Features for Horizon 180 days =====
Graft Loss Training: (12926, 512), Positive: 153, Ratio: 0.0118
Graft Loss Te

In [ ]:
### LOGISTIC REGRESSION

def train_and_eval_logistic(X_train, y_train, X_test, y_test, event_name="Event"):
    clf = LogisticRegression(class_weight={0:1, 1:10},max_iter=1000).fit(X_train, y_train)
    y_pred_proba = clf.predict_proba(X_test)[:, 1]
    
    threshold = .2
    y_pred = (y_pred_proba >= threshold).astype(int)

    #y_pred = clf.predict(X_test)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"\n{event_name} Prediction:")
    print(f"AUC = {roc_auc_score(y_test, y_pred_proba):.4f}")
    print(f"Acc = {accuracy_score(y_test, y_pred):.4f}")
    print(f"Prec = {precision_score(y_test, y_pred):.4f}")
    print(f"Recall (Sensitivity) = {recall_score(y_test, y_pred):.4f}")
    print(f"Specificity = {specificity:.4f}")
    print(f"F1 = {f1_score(y_test, y_pred):.4f}")

for H in horizons:
    train_and_eval_logistic(train_graft_repr[H], train_graft_lbl[H], test_graft_repr[H], test_graft_lbl[H], event_name=f"GraftLoss@{H}")
    #train_and_eval_logistic(train_mort_repr[H], train_mort_lbl[H], test_mort_repr[H], test_mort_lbl[H], event_name=f"Mortality@{H}")
    #train_and_eval_logistic(train_rej_repr[H], train_rej_lbl[H], test_rej_repr[H],  test_rej_lbl[H], event_name=f"Rejection@{H}")



GraftLoss@30 Prediction:
AUC = 0.9473
Acc = 0.9930
Prec = 0.0952
Recall (Sensitivity) = 0.3333
Specificity = 0.9942
F1 = 0.1481

GraftLoss@90 Prediction:
AUC = 0.9648
Acc = 0.9597
Prec = 0.1250
Recall (Sensitivity) = 0.7500
Specificity = 0.9612
F1 = 0.2143

GraftLoss@180 Prediction:
AUC = 0.9650
Acc = 0.9285
Prec = 0.0920
Recall (Sensitivity) = 0.7667
Specificity = 0.9300
F1 = 0.1643


In [7]:
def train_and_eval_mlp(
    X_train, y_train, 
    X_test, y_test, 
    event_name="Event", 
    epochs=50, 
    batch_size=32, 
    lr=5e-3, 
    use_upsampling=True,
    eval_interval=5
):
    # Convert data to tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

    # Create a DataLoader for training
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Define the model
    model_mlp = SimpleMLP(input_dim=X_train.shape[1])
    optimizer = optim.Adam(model_mlp.parameters(), lr=lr, weight_decay=1e-4)

    # Define loss
    if use_upsampling:
        # Example: Increase the loss weight for the positive class
        pos_weight = torch.tensor([25.0])
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    # Evaluation function
    def evaluate_model(epoch=None):
        """Compute metrics on the test set and print them."""
        model_mlp.eval()
        with torch.no_grad():
            logits_test = model_mlp(X_test_t)          # shape (N,)
            probs_test  = torch.sigmoid(logits_test)   # shape (N,)
            y_pred      = (probs_test >= 0.5).int().numpy()
            y_prob      = probs_test.numpy()
            y_true      = y_test_t.numpy()

        # Metrics
        auc_  = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan')
        acc_  = accuracy_score(y_true, y_pred)
        prec_ = precision_score(y_true, y_pred, zero_division=0)
        rec_  = recall_score(y_true, y_pred, zero_division=0)
        f1_   = f1_score(y_true, y_pred, zero_division=0)
        
        # Specificity = TN / (TN + FP)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        # Print
        msg_prefix = f"[{event_name}]"
        if epoch is not None:
            msg_prefix += f" Epoch {epoch}/{epochs}"
        print(
            f"{msg_prefix}\n"
            f"AUC = {auc_:.4f}, Acc = {acc_:.4f}, "
            f"Prec = {prec_:.4f}, Recall = {rec_:.4f}, "
            f"Spec = {specificity:.4f}, F1 = {f1_:.4f}\n"
        )

        model_mlp.train()  # Switch back to training mode

    # Training loop
    model_mlp.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model_mlp(batch_x)    # shape (B,)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}", end='\r')

        # Evaluate every N epochs (e.g., every 5 epochs)
        if epoch % eval_interval == 0 or epoch == epochs:
            # Move to a new line so the loss print isn't overwritten
            print()
            evaluate_model(epoch=epoch)

    # Final model save
    torch.save(model_mlp.state_dict(), f'../models/{event_name}_clf.pth')


# Example usage
for H in horizons:
    train_and_eval_mlp( X_train=train_graft_repr[H], y_train=train_graft_lbl[H], X_test=test_graft_repr[H], y_test=test_graft_lbl[H], # pos weight 25
                       event_name=f"GraftLoss@{H}", epochs=10, batch_size=32, lr=5e-3, use_upsampling=True, eval_interval=2)

    # Rejections
    train_and_eval_mlp(X_train=train_rej_repr[H],y_train=train_rej_lbl[H],X_test=test_rej_repr[H], y_test=test_rej_lbl[H],
                       event_name=f"Rejection@{H}",epochs=10, batch_size=32, lr=5e-3, use_upsampling=True, eval_interval=2) # pos w 100
    
    # Mortality
    train_and_eval_mlp(X_train=train_rej_repr[H],y_train=train_rej_lbl[H],X_test=test_rej_repr[H], y_test=test_rej_lbl[H],
                       event_name=f"Mortality@{H}",epochs=10, batch_size=32, lr=5e-3, use_upsampling=True, eval_interval=2) # pos w 100

Epoch 2/10 - Loss: 0.2436
[GraftLoss@30] Epoch 2/10
AUC = 0.8155, Acc = 0.9982, Prec = 0.0000, Recall = 0.0000, Spec = 1.0000, F1 = 0.0000

Epoch 4/10 - Loss: 0.2130
[GraftLoss@30] Epoch 4/10
AUC = 0.8230, Acc = 0.9982, Prec = 0.0000, Recall = 0.0000, Spec = 1.0000, F1 = 0.0000

Epoch 6/10 - Loss: 0.1959
[GraftLoss@30] Epoch 6/10
AUC = 0.8230, Acc = 0.9982, Prec = 0.0000, Recall = 0.0000, Spec = 1.0000, F1 = 0.0000

Epoch 8/10 - Loss: 0.2517
[GraftLoss@30] Epoch 8/10
AUC = 0.8212, Acc = 0.9982, Prec = 0.0000, Recall = 0.0000, Spec = 1.0000, F1 = 0.0000

Epoch 10/10 - Loss: 0.2692
[GraftLoss@30] Epoch 10/10
AUC = 0.8229, Acc = 0.9982, Prec = 0.0000, Recall = 0.0000, Spec = 1.0000, F1 = 0.0000

Epoch 2/10 - Loss: 0.6317
[Rejection@30] Epoch 2/10
AUC = 0.7675, Acc = 0.9905, Prec = 0.0000, Recall = 0.0000, Spec = 1.0000, F1 = 0.0000

Epoch 4/10 - Loss: 0.6110
[Rejection@30] Epoch 4/10
AUC = 0.7693, Acc = 0.9905, Prec = 0.0000, Recall = 0.0000, Spec = 1.0000, F1 = 0.0000

Epoch 6/10 - Loss:

In [11]:
clf_model = SimpleMLP(input_dim=512)
clf_model.load_state_dict(torch.load("../models/GraftLoss@90_clf.pth", weights_only=True))
clf_model.eval()
clf_model.to(device)

SimpleMLP(
  (fc1): Linear(in_features=512, out_features=128, bias=True)
  (relu): ReLU()
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (relu2): ReLU()
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc3): Linear(in_features=64, out_features=32, bias=True)
  (relu3): ReLU()
  (bn3): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc4): Linear(in_features=128, out_features=1, bias=True)
)

In [ ]:
def train_and_eval_autogluon(X_train, y_train, X_test, y_test, event_name="Event", time_limit=60):
    """
    Train an AutoGluon model and evaluate its performance.
    
    Parameters:
    -----------
    X_train : numpy.ndarray
        Training features
    y_train : numpy.ndarray
        Training labels
    X_test : numpy.ndarray
        Test features
    y_test : numpy.ndarray
        Test labels
    event_name : str
        Name of the event being predicted
    time_limit : int
        Time limit in seconds for AutoGluon training
        
    Returns:
    --------
    predictor : TabularPredictor
        Trained AutoGluon predictor
    metrics : dict
        Performance metrics
    """
    # Convert numpy arrays to pandas DataFrames
    feature_names = [f'feature_{i}' for i in range(X_train.shape[1])]
    train_df = pd.DataFrame(X_train, columns=feature_names)
    test_df = pd.DataFrame(X_test, columns=feature_names)
    
    # Add labels
    train_df['label'] = y_train
    test_df['label'] = y_test
    
    # Check class distribution
    class_counts = np.bincount(y_train)
    print(f"Class distribution in training set: {class_counts}")
    
    # Calculate class weights for imbalance
    pos_scale = (len(y_train) / (2 * np.sum(y_train))) if np.sum(y_train) > 0 else 1.0
    neg_scale = (len(y_train) / (2 * (len(y_train) - np.sum(y_train)))) if len(y_train) - np.sum(y_train) > 0 else 1.0
    
    # Create directory for AutoGluon
    import os
    os.makedirs(f'models/{event_name}', exist_ok=True)
    
    # Initialize and train AutoGluon predictor
    print(f"Training AutoGluon model for {event_name}...")
    predictor = TabularPredictor(
        label='label',
        path=f'models/{event_name}',
        problem_type='binary',
        eval_metric='roc_auc', 
        verbosity=0,
    )
    
    # Train with hyperparameters focused on handling imbalanced data
    predictor.fit(
        train_data=train_df,
        time_limit=time_limit,  # time budget in seconds
        presets='good_quality',
        hyperparameters={
            'GBM': [
                {
                    'extra_trees': True,
                    'scale_pos_weight': pos_scale  # Handle class imbalance
                },
                {
                    'extra_trees': False,
                    'scale_pos_weight': pos_scale  # Handle class imbalance
                }
            ],
            'RF': [
                {
                    'criterion': 'gini',
                    'class_weight': 'balanced'  # Handle class imbalance
                },
                {
                    'criterion': 'entropy',
                    'class_weight': 'balanced'  # Handle class imbalance
                }
            ],
            'XT': [
                {
                    'criterion': 'gini',
                    'class_weight': 'balanced'  # Handle class imbalance
                }
            ],
            'XGB': [
                {
                    'scale_pos_weight': pos_scale,  # Handle class imbalance
                    'max_depth': 6
                }
            ],
            'CAT': [
                {
                    'auto_class_weights': 'Balanced'  # Handle class imbalance
                }
            ],
            'FASTAI': [
                {
                    'weights': pos_scale,  # Handle class imbalance
                    'epochs': 20
                }
            ]
        },
        verbosity=0
    )
    
    # Make predictions on test data
    y_pred_proba = predictor.predict_proba(test_df)
    y_pred_proba_pos = y_pred_proba[1].values if isinstance(y_pred_proba, pd.DataFrame) else y_pred_proba[:, 1]
    
    # Apply threshold of 0.5 for binary prediction
    threshold = 0.5
    y_pred = (y_pred_proba_pos >= threshold).astype(int)
    
    # Calculate metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    # Print metrics
    print(f"\n{event_name} Prediction Results:")
    print(f"AUC = {roc_auc_score(y_test, y_pred_proba_pos):.4f}")
    print(f"Acc = {accuracy_score(y_test, y_pred):.4f}")
    print(f"Prec = {precision_score(y_test, y_pred):.4f}")
    print(f"Recall (Sensitivity) = {recall_score(y_test, y_pred):.4f}")
    print(f"Specificity = {specificity:.4f}")
    print(f"F1 = {f1_score(y_test, y_pred):.4f}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    
    # Save predictor leaderboard
    leaderboard = predictor.leaderboard(test_df, silent=True)
    print("\nModel Leaderboard:")
    print(leaderboard[['model', 'score_val', 'score_test']].head())
    
    # Return predictor and metrics
    metrics = {
        'auc': roc_auc_score(y_test, y_pred_proba_pos),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'specificity': specificity,
        'f1': f1_score(y_test, y_pred)
    }
    
    return predictor, metrics

def optimize_threshold(predictor, X_val, y_val):
    """
    Find optimal classification threshold based on validation data.
    
    Parameters:
    -----------
    predictor : TabularPredictor
        Trained AutoGluon predictor
    X_val : numpy.ndarray
        Validation features
    y_val : numpy.ndarray
        Validation labels
        
    Returns:
    --------
    optimal_threshold : float
        Threshold that maximizes F1 score
    """
    # Convert to DataFrame
    feature_names = [f'feature_{i}' for i in range(X_val.shape[1])]
    val_df = pd.DataFrame(X_val, columns=feature_names)
    
    # Get predictions
    y_pred_proba = predictor.predict_proba(val_df)
    y_pred_proba_pos = y_pred_proba[1].values if isinstance(y_pred_proba, pd.DataFrame) else y_pred_proba[:, 1]
    
    # Try different thresholds
    thresholds = np.linspace(0.1, 0.9, 9)
    f1_scores = []
    
    for threshold in thresholds:
        y_pred = (y_pred_proba_pos >= threshold).astype(int)
        f1 = f1_score(y_val, y_pred)
        f1_scores.append(f1)
    
    # Find threshold with best F1 score
    best_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[best_idx]
    
    print(f"Optimal threshold: {optimal_threshold:.2f} (F1: {f1_scores[best_idx]:.4f})")
    
    return optimal_threshold